# Решения: разбиение выборки

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

In [ ]:
from pathlib import Path
import pandas as pd


DATA_URL = (
    "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/modules/08_04_mnist_knn/data/digits.csv"
)


def find_digits_csv():
    for p in (
        Path("digits.csv"),
        Path("../digits.csv"),
        Path("../../data/digits.csv"),
        Path("../data/digits.csv"),
        Path("../../../data/digits.csv"),
    ):
        if p.exists():
            return p.resolve()
    return DATA_URL


DIGITS_PATH = find_digits_csv()
df = pd.read_csv(DIGITS_PATH)
PIXELS = [c for c in df.columns if c.startswith('p')]

from sklearn.model_selection import train_test_split


## Урок. 1–3. Ручное разбиение

In [ ]:
order = df.sample(frac=1, random_state=0).index
n_train = int(len(df) * 0.75)
train_idx, test_idx = order[:n_train], order[n_train:]
n_overlap = int(train_idx.isin(test_idx).sum())
n_total = len(train_idx) + len(test_idx)
train_df, test_df = df.loc[train_idx], df.loc[test_idx]
share_tr = train_df['label'].value_counts(normalize=True).sort_index()
share_te = test_df['label'].value_counts(normalize=True).sort_index()
max_gap = float((share_tr - share_te).abs().max())
print(len(train_idx), len(test_idx), n_overlap, round(max_gap, 4))

## Урок. 4–5. Библиотека и stratify

In [ ]:
tr, te = train_test_split(df, test_size=0.25, random_state=0)
n_tr_lib, n_te_lib = len(tr), len(te)
tr_s, te_s = train_test_split(df, test_size=0.25, random_state=0, stratify=df['label'])
gap_s = (tr_s['label'].value_counts(normalize=True).sort_index()
         - te_s['label'].value_counts(normalize=True).sort_index()).abs().max()
max_gap_strat = float(gap_s)
print(n_tr_lib, n_te_lib, round(max_gap_strat, 4), round(max_gap, 4))

## Урок. 6–7. Условная доля и крошечная проверка

In [ ]:
n_dark = (df[PIXELS] > 8).sum(axis=1)
dark = df[n_dark > n_dark.median()]
light = df[n_dark <= n_dark.median()]
p_eight_dark = float((dark['label'] == 8).mean())
p_eight_light = float((light['label'] == 8).mean())
COND_NOTE = (
    'Восьмёрка состоит из двух замкнутых петель, чернил на неё уходит больше: '
    'среди «жирных» картинок её доля примерно втрое выше. Знание признака меняет '
    'вероятность цифры — на этом и работает поиск похожих картинок.'
)
tiny = df.sample(20, random_state=11)
p_three_tiny = float((tiny['label'] == 3).mean())
p_three_all = float((df['label'] == 3).mean())
TINY_NOTE = (
    'На 20 картинках одна ошибка сдвигает долю на 5 процентных пунктов: '
    'разница двух распознавателей утонет в случайности выборки.'
)
print(round(p_eight_dark, 3), round(p_eight_light, 3),
      round(p_three_tiny, 3), round(p_three_all, 3))

## Урок. 8–9. Пять разбиений и три части

In [ ]:
spreads = []
for seed in range(5):
    _, te_i = train_test_split(df, test_size=0.25, random_state=seed)
    spreads.append(float((te_i['label'] == 3).mean()))
spread_range = max(spreads) - min(spreads)
SEED_NOTE = (
    'Доля одной цифры в проверочной части гуляет от разбиения к разбиению. '
    'Значит и точность гуляет: сравнивать модели надо на одном фиксированном разбиении.'
)
rest, final = train_test_split(df, test_size=0.2, random_state=0, stratify=df['label'])
fit_part, check_part = train_test_split(rest, test_size=0.25, random_state=0,
                                        stratify=rest['label'])
sizes_three = [len(fit_part), len(check_part), len(final)]
print([round(s, 3) for s in spreads], round(spread_range, 3), sizes_three)

## ДЗ. 1–4

In [ ]:
order1 = df.sample(frac=1, random_state=1).index
mid = len(df) // 2
half_a_idx, half_b_idx = order1[:mid], order1[mid:]
n_overlap = int(half_a_idx.isin(half_b_idx).sum())
p_nine_a = float((df.loc[half_a_idx, 'label'] == 9).mean())
p_nine_b = float((df.loc[half_b_idx, 'label'] == 9).mean())
gap_nine = abs(p_nine_a - p_nine_b)
tr_p, te_p = train_test_split(df, test_size=0.25, random_state=2)
gap_plain = float((tr_p['label'].value_counts(normalize=True).sort_index()
                   - te_p['label'].value_counts(normalize=True).sort_index()).abs().max())
tr_s, te_s = train_test_split(df, test_size=0.25, random_state=2, stratify=df['label'])
gap_strat = float((tr_s['label'].value_counts(normalize=True).sort_index()
                   - te_s['label'].value_counts(normalize=True).sort_index()).abs().max())
STRAT_NOTE = (
    'Когда классов много или какой-то класс редкий, случайное разбиение может '
    'дать в проверочной части почти нет этого класса — оценка станет случайной.'
)
WHY_NOT_TEST = (
    'Проверочные картинки играют роль будущей почты. Если подкручивать распознаватель, '
    'пока не понравится результат именно на них, то мы выбираем настройку под эти 450 конвертов. '
    'На настоящем потоке качество окажется ниже обещанного, а сервис уже подписал договор.'
)
print(n_overlap, round(gap_nine, 3), round(gap_plain, 4), round(gap_strat, 4))